# MART_P08_FLUXO_POLOS

**P08. Qual o fluxo de pacientes do interior de Alagoas para os polos de saúde (Maceió
e Arapiraca)?** *(grafo)*

Cada linha é uma **aresta** município de residência (interior) → polo de atendimento,
por ano, com o volume de internações (SIH) e de procedimentos ambulatoriais aprovados (SIA).
`PCT_INTERNACOES_DESTINO` = participação dessa origem no total de internações do polo.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P08_FLUXO_POLOS
    (NR_ANO, COD_IBGE_ORIGEM, MUNICIPIO_ORIGEM, REGIAO_ORIGEM, COD_IBGE_DESTINO,
     MUNICIPIO_DESTINO, QTD_INTERNACOES, QTD_AMBULATORIAL, PCT_INTERNACOES_DESTINO)
WITH hosp AS (
    SELECT t.NR_ANO, f.SK_MUNICIPIO_RESIDENCIA AS ORIGEM, f.SK_MUNICIPIO_ATENDIMENTO AS DESTINO,
           SUM(f.QTD_INTERNACAO) AS QTD
    FROM FATO_INTERNACAO f
    JOIN DIM_TEMPO t ON t.SK_TEMPO = f.SK_TEMPO_INTERNACAO
    GROUP BY t.NR_ANO, f.SK_MUNICIPIO_RESIDENCIA, f.SK_MUNICIPIO_ATENDIMENTO
),
amb AS (
    SELECT t.NR_ANO, f.SK_MUNICIPIO_RESIDENCIA AS ORIGEM, f.SK_MUNICIPIO_ATENDIMENTO AS DESTINO,
           SUM(f.QTD_APROVADA) AS QTD
    FROM FATO_PRODUCAO_AMBULATORIAL f
    JOIN DIM_TEMPO t ON t.SK_TEMPO = f.SK_TEMPO_COMPETENCIA
    GROUP BY t.NR_ANO, f.SK_MUNICIPIO_RESIDENCIA, f.SK_MUNICIPIO_ATENDIMENTO
),
total_destino AS (
    SELECT NR_ANO, DESTINO, SUM(QTD) AS TOTAL FROM hosp GROUP BY NR_ANO, DESTINO
),
fluxo AS (
    SELECT COALESCE(h.NR_ANO, a.NR_ANO) AS NR_ANO, COALESCE(h.ORIGEM, a.ORIGEM) AS ORIGEM,
           COALESCE(h.DESTINO, a.DESTINO) AS DESTINO, NVL(h.QTD, 0) AS QTD_H, NVL(a.QTD, 0) AS QTD_A
    FROM hosp h
    FULL OUTER JOIN amb a ON a.NR_ANO = h.NR_ANO AND a.ORIGEM = h.ORIGEM AND a.DESTINO = h.DESTINO
)
SELECT x.NR_ANO, mo.COD_IBGE_6, mo.NOME_MUNICIPIO, mo.REGIAO_SAUDE, md.COD_IBGE_6,
       md.NOME_MUNICIPIO, x.QTD_H, x.QTD_A,
       ROUND(100 * x.QTD_H / NULLIF(td.TOTAL, 0), 2)
FROM fluxo x
JOIN DIM_MUNICIPIO mo     ON mo.SK_MUNICIPIO = x.ORIGEM
JOIN DIM_MUNICIPIO md     ON md.SK_MUNICIPIO = x.DESTINO
LEFT JOIN total_destino td ON td.NR_ANO = x.NR_ANO AND td.DESTINO = x.DESTINO
WHERE mo.TIPO_LOCALIDADE = 'INTERIOR'
  AND md.COD_IBGE_6 IN ('270430', '270030')
  AND mo.SK_MUNICIPIO <> md.SK_MUNICIPIO
  AND x.NR_ANO IN (SELECT DISTINCT TO_NUMBER(SUBSTR(ANO_MES_CMPT, 1, 4)) FROM FATO_INTERNACAO)
"""

In [ ]:
recarregar("MART_P08_FLUXO_POLOS", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P08_FLUXO_POLOS FETCH FIRST 10 ROWS ONLY")